<a href="https://colab.research.google.com/github/Akhilesh-Chandewar/VisualQuest/blob/main/VisualQuest_ResNET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from numpy import linalg as LA

from tensorflow.keras.applications.resnet50  import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model

In [ ]:
class getResNet50Model:
    def __init__(self):
        # weights: 'imagenet'
        # pooling: 'max' or 'avg'
        # input_shape: (width, height, 3), width and height should >= 48
        self.input_shape = (224, 224, 3)
        self.resnet_model = ResNet50(weights='imagenet', input_shape=self.input_shape, include_top = True)
        self.output = self.resnet_model.get_layer('avg_pool').output
        self.resnet_model = Model(self.resnet_model.input, self.output)
        #self.resnet_model.summary()


    '''
    Use Resnet50 model to extract features
    Output normalized feature vector
    '''
    def extract_feat(self, img_path):
        img = image.load_img(img_path, target_size=(self.input_shape[0], self.input_shape[1]))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feat = self.resnet_model.predict(img)
        norm_feat = feat[0]/LA.norm(feat[0])
        return norm_feat

In [ ]:
import os
import h5py

In [ ]:
images_path ="all_images/"
img_list = [os.path.join(images_path,f) for f in os.listdir(images_path)]
img_list

['all_images/4.png',
 'all_images/horse2.jpg',
 'all_images/zebra2.jpg',
 'all_images/chihuahua.JPG',
 'all_images/irish_terrier.JPG',
 'all_images/1.png',
 'all_images/horse1.jpg',
 'all_images/2.png',
 'all_images/tiger1.jpg',
 'all_images/monkey1.jpg',
 'all_images/monkey2.jpg',
 'all_images/zebra1.jpg',
 'all_images/tiger2.jpg']

In [ ]:
model = getResNet50Model()

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
path = "all_images/"

feats = []
names = []

for im in os.listdir(path):  #iterate through all images to extract features
    print("Extracting features from image - ", im)
    X = model.extract_feat(path+im)

    feats.append(X)
    names.append(im)

feats = np.array(feats)

# directory for storing extracted features
output = "ResnetFeatures.h5"

print(" writing feature extraction results to h5 file")


h5f = h5py.File(output, 'w')
h5f.create_dataset('dataset_1', data = feats)
h5f.create_dataset('dataset_2', data = np.string_(names))
h5f.close()

Extracting features from image -  4.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Extracting features from image -  horse2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step
Extracting features from image -  zebra2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step
Extracting features from image -  chihuahua.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 348ms/step
Extracting features from image -  irish_terrier.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
Extracting features from image -  1.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
Extracting features from image -  horse1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
Extracting features from image -  2.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
Extracting features from image -  tiger1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
Extracting features from image -  monkey1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
Extracting features from image -  monkey2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
Extracting features from image -  zebra1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/ste

In [9]:
h5f = h5py.File("ResnetFeatures.h5",'r')
feats = h5f['dataset_1'][:]
imgNames = h5f['dataset_2'][:]
h5f.close()

In [14]:
queryImg = "query_images/histo.jpg"

In [15]:
X = model.extract_feat(queryImg)
len(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step


2048

In [16]:
scores = []
from scipy import spatial
for i in range(feats.shape[0]):
    score = 1-spatial.distance.cosine(X, feats[i])
    scores.append(score)
scores = np.array(scores)
rank_ID = np.argsort(scores)[::-1]
rank_score = scores[rank_ID]

In [17]:
maxres = 3
imlist = [imgNames[index] for i,index in enumerate(rank_ID[0:maxres])]
print("top %d images in order are: " %maxres, imlist)

top 3 images in order are:  [b'4.png', b'2.png', b'1.png']
